In [1]:
from pathlib import Path
import re
import polars as pl

BASE_DIR = Path("/media/do-ald533/HD1/github/llm_argumentation_structuring")
GOLD_DIR = BASE_DIR / "data" / "Golden Standard"
OUT_DIR = BASE_DIR / "output"

DATASETS = ["AAEC", "AbstRCT"]
COMPONENT_THRESHOLD = 0.80
RELATION_THRESHOLD = 0.80

# Validation Notebook

This notebook evaluates predicted argument components and relations against gold standard files in the local project.

It reports exact and fuzzy metrics for AAEC and AbstRCT.

Fuzzy matching rule:
- A prediction matches a gold item if token containment is above a threshold.
- This allows near matches instead of requiring 100% identical text.

In [2]:
def detect_separator(path: Path) -> str:
    """Detect CSV separator using the header line."""
    header = path.read_text(encoding="utf-8").splitlines()[0]
    return ";" if header.count(";") > header.count(",") else ","


def read_csv_auto(path: Path) -> pl.DataFrame:
    """Read CSV with delimiter detection."""
    return pl.read_csv(path, separator=detect_separator(path))


def normalize_text(text: str) -> str:
    """Normalize text for robust matching."""
    if text is None:
        return ""
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def token_set(text: str) -> set[str]:
    """Convert text to a token set after normalization."""
    return set(normalize_text(text).split())


def containment_ratio(gold_text: str, pred_text: str) -> float:
    """How much of gold tokens are contained in prediction tokens."""
    g = token_set(gold_text)
    p = token_set(pred_text)
    if not g:
        return 0.0
    return len(g & p) / len(g)


def safe_div(num: float, den: float) -> float:
    """Safe division."""
    return num / den if den else 0.0


def f1_score(precision: float, recall: float) -> float:
    """F1 from precision and recall."""
    return safe_div(2 * precision * recall, precision + recall)

In [3]:
def match_rows(
    gold_df: pl.DataFrame,
    pred_df: pl.DataFrame,
    text_cols: list[str],
    use_label: bool,
    threshold: float,
) -> dict:
    """Greedy one-to-one matching between gold and prediction rows using containment."""
    keys = ["text_id", "labels"] if use_label else ["text_id"]

    gold_rows = gold_df.to_dicts()
    pred_rows = pred_df.to_dicts()

    pred_index: dict[tuple, list[int]] = {}
    for pi, p in enumerate(pred_rows):
        k = tuple(p[key] for key in keys)
        pred_index.setdefault(k, []).append(pi)

    candidates: list[tuple[float, int, int]] = []

    for gi, g in enumerate(gold_rows):
        key = tuple(g[k] for k in keys)
        for pi in pred_index.get(key, []):
            p = pred_rows[pi]
            scores = [containment_ratio(g[col], p[col]) for col in text_cols]
            score = sum(scores) / len(scores)
            if score >= threshold:
                candidates.append((score, gi, pi))

    candidates.sort(reverse=True, key=lambda x: x[0])

    used_gold = set()
    used_pred = set()
    tp = 0

    for score, gi, pi in candidates:
        if gi in used_gold or pi in used_pred:
            continue
        used_gold.add(gi)
        used_pred.add(pi)
        tp += 1

    fp = len(pred_rows) - tp
    fn = len(gold_rows) - tp

    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1_score(precision, recall),
    }


def evaluate_dataset(
    dataset: str,
    component_threshold: float = COMPONENT_THRESHOLD,
    relation_threshold: float = RELATION_THRESHOLD,
) -> pl.DataFrame:
    """Evaluate one dataset and return metrics as a Polars DataFrame."""
    gc = GOLD_DIR / f"components_{dataset}.csv"
    gr = GOLD_DIR / f"relations_{dataset}.csv"
    pc = OUT_DIR / f"components_{dataset}.csv"
    pr = OUT_DIR / f"relations_{dataset}.csv"

    required = [gc, gr, pc, pr]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError(f"Missing files for {dataset}: {missing}")

    comp_gold = read_csv_auto(gc).select(["text_id", "component_tokens", "labels"])
    comp_pred = read_csv_auto(pc).select(["text_id", "component_tokens", "labels"])
    rel_gold = read_csv_auto(gr).select(["text_id", "source_tokens", "target_tokens", "labels"])
    rel_pred = read_csv_auto(pr).select(["text_id", "source_tokens", "target_tokens", "labels"])

    comp_gold = comp_gold.with_columns(
        pl.col("component_tokens").map_elements(normalize_text, return_dtype=pl.String),
        pl.col("labels").map_elements(normalize_text, return_dtype=pl.String),
    )
    comp_pred = comp_pred.with_columns(
        pl.col("component_tokens").map_elements(normalize_text, return_dtype=pl.String),
        pl.col("labels").map_elements(normalize_text, return_dtype=pl.String),
    )

    rel_gold = rel_gold.with_columns(
        pl.col("source_tokens").map_elements(normalize_text, return_dtype=pl.String),
        pl.col("target_tokens").map_elements(normalize_text, return_dtype=pl.String),
        pl.col("labels").map_elements(normalize_text, return_dtype=pl.String),
    )
    rel_pred = rel_pred.with_columns(
        pl.col("source_tokens").map_elements(normalize_text, return_dtype=pl.String),
        pl.col("target_tokens").map_elements(normalize_text, return_dtype=pl.String),
        pl.col("labels").map_elements(normalize_text, return_dtype=pl.String),
    )

    rows = []
    rows.append({"dataset": dataset, "metric": "component_span_exact", **match_rows(comp_gold, comp_pred, ["component_tokens"], False, 1.0)})
    rows.append({"dataset": dataset, "metric": "component_span_fuzzy", **match_rows(comp_gold, comp_pred, ["component_tokens"], False, component_threshold)})
    rows.append({"dataset": dataset, "metric": "component_label_exact", **match_rows(comp_gold, comp_pred, ["component_tokens"], True, 1.0)})
    rows.append({"dataset": dataset, "metric": "component_label_fuzzy", **match_rows(comp_gold, comp_pred, ["component_tokens"], True, component_threshold)})

    rows.append({"dataset": dataset, "metric": "relation_link_exact", **match_rows(rel_gold, rel_pred, ["source_tokens", "target_tokens"], False, 1.0)})
    rows.append({"dataset": dataset, "metric": "relation_link_fuzzy", **match_rows(rel_gold, rel_pred, ["source_tokens", "target_tokens"], False, relation_threshold)})
    rows.append({"dataset": dataset, "metric": "relation_label_exact", **match_rows(rel_gold, rel_pred, ["source_tokens", "target_tokens"], True, 1.0)})
    rows.append({"dataset": dataset, "metric": "relation_label_fuzzy", **match_rows(rel_gold, rel_pred, ["source_tokens", "target_tokens"], True, relation_threshold)})

    return pl.DataFrame(rows)

In [4]:
all_results = [evaluate_dataset(ds) for ds in DATASETS]

results = pl.concat(all_results).select(
    ["dataset", "metric", "tp", "fp", "fn", "precision", "recall", "f1"]
).sort(["dataset", "metric"])

results

summary = results.pivot(values="f1", index="metric", on="dataset").sort("metric")
summary

metric,AAEC,AbstRCT
str,f64,f64
"""component_label_exact""",0.510266,0.177215
"""component_label_fuzzy""",0.512548,0.180294
"""component_span_exact""",0.862357,0.520356
"""component_span_fuzzy""",0.863878,0.523435
"""relation_label_exact""",0.183436,0.160641
"""relation_label_fuzzy""",0.225153,0.163471
"""relation_link_exact""",0.188344,0.174159
"""relation_link_fuzzy""",0.233742,0.179818
